# 13 · M1.8 — M1.7 with the width fixed at long horizons: a zero floor and the variance ratio

## Executive summary (read this first)

**The problem.** On the backtest M1.5's 90 % interval caught the answer only 83 % of the time on rates at 126 days and 75 % at
189+ days, and 84 % / 81 % on FX. The walk assumes moves add up like √h: h steps are √h times as wide as one. Rates do not
behave like that. They trend through policy cycles, and at zero they cannot keep falling.

**M1.8 = M1.7 + two changes to the width:**

| piece | what it does | why |
|---|---|---|
| **zero floor** | a Treasury-yield draw below 0 % is set to 0 % | US yields stayed at or above 0 in 2009–15 and 2020–21, while the walk put many draws below it |
| **variance ratio** | sd × √VR for each asset and horizon | VR = mean(h-step move²) ÷ (h × mean(1-step move²)) over the asset's whole history before the date: 1 for a random walk, above 1 when moves keep going (cycles), below 1 when they cancel. Shrunk toward 1: VR = 1 + (VR_raw − 1) × n ÷ (n + 30), n = history ÷ h |

This is the classic variance-ratio test of Lo and MacKinlay used as a correction. Moves are not demeaned, except for the CPI and
NFP assets that carry a drift. A trend the model does not forecast is width.

**Results** (§3): M1.8 **0.9266** × M0 (M1.7 0.9312): the floor gives 0.9278, then the variance ratio 0.9266. F1 gains most
(0.883 → 0.871): its long-horizon rates cards include the zero-rate years. The floor is set in advance. The shrinkage
constant 30 was chosen over 10 in a scratch test on M1.5 (0.9422 vs 0.9429).

§2 works both pieces through with numbers. No realized outcome is read.

## 0 — Setup (as notebook 07)

In [1]:
import bisect, multiprocessing as mp, pathlib, sys, time, tomllib, warnings, zlib
import numpy as np, pandas as pd, pyarrow.parquet as pq

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "model_experiment_v3" else pathlib.Path.cwd() / "model_experiment_v3"
REPO, UNITS, DATA = HERE.parent, HERE.parent / "units", HERE / "data"
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
import m0_baseline as m0
from qfbench2_track_forecasting import scoring as S
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
WORKERS = max(1, (mp.cpu_count() or 2) - 2)
geo = lambda r: float(np.exp(np.log(np.asarray(r, float)).mean()))

ANSWERS = pd.read_parquet(DATA / "backtest_answers.parquet")
M0_SCORES = pd.read_parquet(DATA / "m0_backtest_scores.parquet")
M11_SCORES = pd.read_parquet(DATA / "m1_1_backtest_scores.parquet")
UNIT_IDS = sorted(ANSWERS.unit.unique())
RULES = tomllib.loads((HERE / "transformations.toml").read_text())
SH = RULES["shocks"]
WINDOW = 260

def asset_type(a):
    return RULES["assets"].get(a, "other")

def rule_for(asset, target_type):
    """card.toml's target type + the asset -> {transform, shock, drift} (transformations.toml)."""
    t = RULES["assets"].get(asset)
    base = RULES["types"][t] if t else {"transform": RULES["default"]["level"], "shock": RULES["default"]["shock"],
                                        "drift": RULES["default"]["drift"]}
    tr = RULES["default"]["log_return"] if target_type == "log_return" else base["transform"]
    return {"transform": tr, "shock": base["shock"], "drift": bool(base["drift"])}

def load_unit(u):
    ud = UNITS / u
    card = tomllib.loads((ud / "card.toml").read_text()); t = card["targets"]
    assets, horizons = [str(a) for a in t["asset_ids"]], [int(h) for h in t["horizons"]]
    uid, asof, tt = card["task"]["id"], card["provenance"]["data_cutoff"], t.get("target_type", "level")
    rows = {}
    for a in assets:
        for path in sorted(ud.glob("*.parquet")):
            tab = pq.read_table(path).to_pydict()
            acol = "asset" if "asset" in tab else ("asset_id" if "asset_id" in tab else None)
            if acol is None:
                continue
            r = sorted((str(d)[:10], float(v)) for d, x, v in zip(tab["date"], tab[acol], tab["value"])
                       if str(x) == a and str(d)[:10] <= asof and v is not None)
            if r:
                rows[a] = r; break
    return {"unit": u, "id": uid, "family": card["metadata"]["category"], "card": card, "assets": assets,
            "horizons": horizons, "target_type": tt, "asof": asof, "rows": rows, "dates": {a: [d for d, _ in rows[a]] for a in assets},
            "rules": {a: rule_for(a, tt) for a in assets}, "steps": {h: m0._MONTHLY_STEPS.get(uid, {}).get(h, h) for h in horizons}}

UNITS_DATA = {u: load_unit(u) for u in UNIT_IDS}
TYPE = {u: "+".join(sorted({asset_type(a) for a in U["assets"]})) for u, U in UNITS_DATA.items()}
BACKTEST = {}
for u, g in ANSWERS.groupby("unit"):
    U = UNITS_DATA[u]
    order = {(a, h): i for i, (a, h) in enumerate((a, h) for a in U["assets"] for h in U["horizons"])}
    BACKTEST[u] = [(o, gg.assign(i=[order[(a, h)] for a, h in zip(gg.asset, gg.horizon)]).sort_values("i").answer.to_numpy())
                   for o, gg in g.groupby("origin")]
print(f"{len(UNIT_IDS)} units, {sum(len(v) for v in BACKTEST.values()):,} backtest dates")
display(pd.DataFrame({t: {"transform": v["transform"], "shock": v["shock"], "drift": v["drift"]} for t, v in RULES["types"].items()}).T)
print("Student-t:", SH)

103 units, 23,635 backtest dates


,transform,shock,drift
rates,diff,student_t,False
fx,log_diff,student_t,False
cpi,log_diff,normal,True
payrolls,diff,normal,True
unemployment,diff,normal,False
equity_factor,as_return,student_t,False


Student-t: {'nu': 'fit', 'nu_min': 4.2, 'nu_max': 30.0, 'normal_below_kurtosis': 0.2}


## 1 — The model: notebook 12's M1.7, plus the floor and the variance ratio

`m18_draws(U, origin)` is M1.8; `floor=False, vr=False` is M1.7, and the check below asserts it reproduces notebook 12's saved
scores exactly. `vr=False` alone shows what the floor does by itself.

In [2]:
def fit_nu(x, rows_t):
    """nu from the window's excess kurtosis k: a Student-t has k = 6 / (nu - 4), so nu = 4 + 6 / k."""
    if not rows_t:
        return None
    z = x[rows_t] - x[rows_t].mean(axis=1, keepdims=True)
    k = float(np.mean((z ** 4).mean(axis=1) / (z ** 2).mean(axis=1) ** 2 - 3))
    return None if k < SH["normal_below_kurtosis"] else float(np.clip(4 + 6 / k, SH["nu_min"], SH["nu_max"]))

def fit(U, origin, steps_window=None):
    """The window's steps per asset -> (assets sorted, steps matrix x, log mode, last value)."""
    by_asset = sorted(U["assets"]); steps, log_mode, last = {}, {}, {}
    for a in by_asset:
        k = bisect.bisect_right(U["dates"][a], origin)
        rows = U["rows"][a][max(0, k - (m0._WINDOW if steps_window is None else steps_window + 1)): k]
        log_mode[a] = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
        if log_mode[a]:
            steps[a], _ = m0._steps([(d, np.log(v)) for d, v in rows], "level"); last[a] = rows[-1][1]
        else:
            steps[a], lv = m0._steps(rows, U["target_type"]); last[a] = lv if U["target_type"] == "level" else 0.0
    common = sorted(set.intersection(*(set(steps[a]) for a in by_asset)))
    x = np.array([[steps[a][d] for d in common] for a in by_asset])
    return by_asset, x, log_mode, last

def ewma_sd_asset(U, a, origin, halflife):
    """EWMA sd of one asset's steps (its own transform) over the last 8 x halflife steps: weight 0.5 ** (age / halflife)."""
    k = bisect.bisect_right(U["dates"][a], origin)
    rows = U["rows"][a][max(0, k - (int(8 * halflife) + 1)): k]
    lm = U["rules"][a]["transform"] == "log_diff" and min(v for _, v in rows) > 0
    st, _ = m0._steps([(d, np.log(v)) for d, v in rows] if lm else rows, "level" if lm else U["target_type"])
    v = np.array([st[d] for d in sorted(st)])
    age = np.arange(len(v))[::-1]; w = 0.5 ** (age / halflife); w = w / w.sum()
    m = w @ v
    return float(np.sqrt(w @ (v - m) ** 2))

# Every asset's whole history as steps (its own transform, M0's gap rule), computed once: the recent trend (M1.7) looks
# back 126 steps, the variance ratio (M1.8) over the whole history.
FULL = {}
for _u, _U in UNITS_DATA.items():
    for _a in _U["assets"]:
        _rows = _U["rows"][_a]
        _lm = _U["rules"][_a]["transform"] == "log_diff" and min(v for _, v in _rows) > 0
        _st, _ = m0._steps([(d, np.log(v)) for d, v in _rows] if _lm else _rows, "level" if _lm else _U["target_type"])
        _ds = sorted(_st)
        FULL[(_u, _a)] = (np.array([str(d) for d in _ds]), np.array([_st[d] for d in _ds]))

def hist_steps(U, a, origin):
    """All of one asset's steps up to the origin, oldest first."""
    d, v = FULL[(U["unit"], a)]
    return v[: np.searchsorted(d, origin, side="right")]

def recent_trend(U, a, origin, mu_window):
    """M1.7's drift per step.
    CPI, NFP (drift by type): the mean of the last 12 monthly steps -- this year's trend, not the 21-year average.
    Everything else: the mean m of the last 126 steps, shrunk toward 0 by its t-statistic t = m / (sd / sqrt(126))
    (James-Stein): drift = m x max(0, 1 - 4 / t^2). |t| <= 2 gives no drift at all; |t| = 4 keeps 75 %."""
    v = hist_steps(U, a, origin)
    if U["rules"][a]["drift"]:
        return float(v[-12:].mean()) if len(v) >= 12 else mu_window
    v = v[-126:]
    sd = v.std(ddof=1)
    t = v.mean() / (sd / np.sqrt(len(v))) if sd > 0 else 0.0
    return float(v.mean() * max(0.0, 1.0 - 4.0 / max(t * t, 1e-12)))

def variance_ratio(U, a, h_steps, origin, n0=30.0):
    """M1.8: VR = mean(h-step move^2) / (h x mean(1-step move^2)) over the asset's whole history before the origin.
    VR > 1: moves add up faster than sqrt(h) (trends, cycles); VR < 1: they partly cancel (mean reversion). Raw moves (no
    demeaning) for assets without a drift, because a trend we do not forecast is width. Shrunk toward 1 by the number of
    independent windows: VR = 1 + (VR_raw - 1) x n / (n + 30), n = history steps / h."""
    v = hist_steps(U, a, origin)
    if h_steps <= 1 or len(v) < 2 * h_steps + 20:
        return 1.0
    m = v.mean() if U["rules"][a]["drift"] else 0.0
    c = np.concatenate([[0.0], np.cumsum(v - m)])
    ratio = np.mean((c[h_steps:] - c[:-h_steps]) ** 2) / (h_steps * np.mean((v - m) ** 2))
    n_eff = len(v) / h_steps
    return 1.0 + (ratio - 1.0) * n_eff / (n_eff + n0)

def m18_draws(U, origin, drift17=True, floor=True, vr=True, ewma_halflife=63):
    """M1.8 = M1.7 + a zero floor on Treasury yields (floor) + the variance ratio on the width (vr).
    floor=False, vr=False is M1.7 (notebook 12)."""
    N = m0.N_DRAWS
    by_asset, x, log_mode, last = fit(U, origin, WINDOW)
    assets, horizons = U["assets"], U["horizons"]
    mu, sigma = x.mean(axis=1), np.atleast_2d(np.cov(x))
    sd_w = np.sqrt(np.diag(sigma)); corr = np.nan_to_num(sigma / np.outer(sd_w, sd_w), nan=0.0); np.fill_diagonal(corr, 1.0)
    sd_e = np.array([ewma_sd_asset(U, a, origin, ewma_halflife) for a in by_asset])     # M1.5: EWMA sd, halflife 63
    sigma = corr * np.outer(sd_e, sd_e)
    ai = {a: i for i, a in enumerate(by_asset)}
    cells = [(a, h) for a in by_asset for h in sorted(horizons)]; s = U["steps"]
    drift = {a: (mu[ai[a]] if U["rules"][a]["drift"] else 0.0) for a in by_asset}      # M1.5: drift by type
    if drift17:                                                                          # M1.7: the recent trend
        drift = {a: recent_trend(U, a, origin, mu[ai[a]]) for a in by_asset}
    def centre(a, h, d):
        return (0.0 if log_mode[a] else last[a]) + s[h] * d[a]
    mean = np.array([centre(a, h, drift) for a, h in cells])
    cov = np.array([[min(s[h1], s[h2]) * sigma[ai[a1], ai[a2]] for a2, h2 in cells] for a1, h1 in cells])
    cov[np.diag_indices_from(cov)] += 1e-10; j = cov + 1e-9 * np.eye(len(cells))
    try: f = np.linalg.cholesky(j)
    except np.linalg.LinAlgError: f = np.diag(np.sqrt(np.diag(j)))
    seed = zlib.crc32(U["id"].encode()) & 0x7FFFFFFF
    dev = np.random.default_rng(seed).standard_normal((N, len(cells))) @ f.T
    scale = np.ones(len(cells))
    if vr:
        scale = scale * np.array([np.sqrt(variance_ratio(U, a, s[h], origin)) for a, h in cells])
    dev = dev * scale
    t_cells = np.array([U["rules"][a]["shock"] == "student_t" for a, h in cells])
    nu = fit_nu(x, [ai[a] for a in by_asset if U["rules"][a]["shock"] == "student_t"])
    if nu is not None and t_cells.any():
        dev = dev * np.where(t_cells, np.sqrt((nu - 2.0) / np.random.default_rng([seed, 7]).chisquare(nu, size=(N, 1))), 1.0)
    sm = mean + dev
    out = np.empty((N, len(assets), len(horizons)))
    for ci, (a, h) in enumerate(cells):
        col = last[a] * np.exp(sm[:, ci]) if log_mode[a] else sm[:, ci]
        if floor and asset_type(a) == "rates":
            col = np.maximum(col, 0.0)          # zero lower bound on Treasury yields
        if s[h] <= 0:
            col = np.full(N, U["rows"][a][bisect.bisect_right(U["dates"][a], origin) - 1][1])
        out[:, assets.index(a), horizons.index(h)] = col
    return out

cats = ["marginal", "joint", "tail"]

def parts(card, samples, y):
    p = card.get("scoring", {}).get("params", {})
    return S._composite(samples, y, weights=m0.card_weights(card, samples.shape[1]),
                        tail_levels=tuple(p.get("tail_levels", (0.01, 0.05, 0.95, 0.99))), joint=S.card_joint_statistic(card),
                        tail_metric=str(p.get("tail_metric", S.DEFAULT_TAIL_METRIC)), ref_scale=None)
mean0 = M0_SCORES.groupby("unit")[cats].mean()
def weights(u):
    U = UNITS_DATA[u]; return m0.card_weights(U["card"], len(U["assets"]) * len(U["horizons"]))
def vs_m0(scores):
    m = scores.groupby("unit")[cats].mean()
    return pd.Series({u: sum(wk * m.loc[u, k] / mean0.loc[u, k] for k, wk in zip(cats, weights(u)) if wk) for u in UNIT_IDS})

def draw_variant(U, o, kw):
    return m18_draws(U, o, **kw)
VARIANTS = {"M1.7": dict(floor=False, vr=False), "M1.7 + floor": dict(vr=False), "M1.8": {}}
def score_unit(u):
    U = UNITS_DATA[u]
    out = []
    for o, y in BACKTEST[u]:
        for m, kw in VARIANTS.items():
            d = draw_variant(U, o, kw)
            out.append({"unit": u, "family": U["family"], "origin": o, "model": m, **parts(U["card"], d.reshape(d.shape[0], -1), y)})
    return out
t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    G = pd.DataFrame([r for rows in pool.map(score_unit, UNIT_IDS, chunksize=1) for r in rows])
print(f"scored {len(G) // len(VARIANTS):,} dates x {len(VARIANTS)} variants in {time.time() - t0:.0f}s")
SCV = {m: vs_m0(G[G.model == m]) for m in VARIANTS}
d = float((SCV["M1.7"] - vs_m0(pd.read_parquet(DATA / "m1_7_backtest_scores.parquet"))).abs().max())
print(f"check — the new pieces switched off vs notebook 12's saved M1.7: max |difference| per unit = {d:.1e}")
assert d < 1e-12
print(pd.Series({m: s.mean() for m, s in SCV.items()}, name="competition score vs M0 (all units)").round(4).to_string())
M18 = G[G.model == "M1.8"].drop(columns="model").reset_index(drop=True)

scored 23,635 dates x 3 variants in 8s
check — the new pieces switched off vs notebook 12's saved M1.7: max |difference| per unit = 0.0e+00
M1.7           0.9312
M1.7 + floor   0.9278
M1.8           0.9266


## 2 — Worked example: the floor and the variance ratio

(a) The floor on a zero-rate date: the 2-year yield in late 2010, at 0.4 %. (b) The variance ratio of the 2-year yield, step
by step, on two dates: the zero-rate years and the 2023 hiking cycle. (c) The average √VR multiplier by asset type and horizon.

In [3]:
def last_value(U, a, o):
    return U["rows"][a][bisect.bisect_right(U["dates"][a], o) - 1][1]

u = "t2-F1-liftoff-telegraph-2015"; U = UNITS_DATA[u]; o, y = BACKTEST[u][0]
before = m18_draws(U, o, floor=False, vr=False)[:, 0, :]; after = m18_draws(U, o, vr=False)[:, 0, :]
display(pd.DataFrame({"horizon": U["horizons"], "last value": last_value(U, "UST_2Y", o),
                      "M1.7: share of draws below 0 %": (before < 0).mean(axis=0), "M1.7: lowest draw": before.min(axis=0),
                      "with the floor: share below 0 %": (after < 0).mean(axis=0), "answer": y}).set_index("horizon")
        .rename_axis(f"(a) {u}, UST_2Y, {o}"))

def vr_steps(U, a, o, hs):
    v = hist_steps(U, a, o); rows = {}
    for h in hs:
        c = np.concatenate([[0.0], np.cumsum(v)]); mv = c[h:] - c[:-h]
        raw = np.mean(mv ** 2) / (h * np.mean(v ** 2)); n = len(v) / h; vr = 1 + (raw - 1) * n / (n + 30)
        rows[h] = {"history steps": len(v), "mean(1-step move²)": np.mean(v ** 2), "mean(h-step move²)": np.mean(mv ** 2),
                   "VR raw = h-step ÷ (h × 1-step)": raw, "n = history ÷ h": n, "VR = 1 + (raw - 1) × n / (n + 30)": vr,
                   "sd multiplier √VR": np.sqrt(vr), "check: variance_ratio()": variance_ratio(U, a, h, o)}
    return pd.DataFrame(rows)
for u, o in (("t2-F1-liftoff-telegraph-2015", BACKTEST["t2-F1-liftoff-telegraph-2015"][0][0]),
             ("t2-F1-greater-confidence-2024", BACKTEST["t2-F1-greater-confidence-2024"][-1][0])):
    print(f"(b) {u}, UST_2Y, variance ratio at {o}:"); display(vr_steps(UNITS_DATA[u], "UST_2Y", o, [126, 189]))

rec = []
for u in UNIT_IDS:
    U = UNITS_DATA[u]
    for o, _ in BACKTEST[u][::10]:
        for a in U["assets"]:
            for h in U["horizons"]:
                rec.append({"asset type": asset_type(a), "horizon": h if U["steps"][h] == h else f"{h} ({U['steps'][h]} months)",
                            "sqrt VR": np.sqrt(variance_ratio(U, a, U["steps"][h], o))})
print("(c) the average sd multiplier √VR on the backtest (every 10th date):")
pd.DataFrame(rec).groupby(["asset type", "horizon"])["sqrt VR"].agg(["mean", "min", "max", "size"])

,last value,M1.7: share of draws below 0 %,M1.7: lowest draw,with the floor: share below 0 %,answer
"(a) t2-F1-liftoff-telegraph-2015, UST_2Y, 2010-09-01",,,,,
126,0.5,0.098,-0.654,0,0.7
189,0.5,0.152,-1.127,0,0.43


(b) t2-F1-liftoff-telegraph-2015, UST_2Y, variance ratio at 2010-09-01:


,126,189
history steps,2669,2669
mean(1-step move²),0.004258,0.004258
mean(h-step move²),0.8243,1.383
VR raw = h-step ÷ (h × 1-step),1.536,1.719
n = history ÷ h,21.18,14.12
VR = 1 + (raw - 1) × n / (n + 30),1.222,1.23
sd multiplier √VR,1.105,1.109
check: variance_ratio(),1.222,1.23


(b) t2-F1-greater-confidence-2024, UST_2Y, variance ratio at 2023-04-28:


,126,189
history steps,5834,5834
mean(1-step move²),0.002889,0.002889
mean(h-step move²),0.6237,1.09
VR raw = h-step ÷ (h × 1-step),1.713,1.997
n = history ÷ h,46.3,30.87
VR = 1 + (raw - 1) × n / (n + 30),1.433,1.506
sd multiplier √VR,1.197,1.227
check: variance_ratio(),1.433,1.506


(c) the average sd multiplier √VR on the backtest (every 10th date):


mean    min    max  size
asset type    horizon                                  
cpi           140 (8 months)  1.201   1.13  1.271     2
              160 (9 months)   1.19  1.115  1.264     2
              21 (2 months)   1.156  1.156  1.156     1
equity_factor 7               1.144    1.1  1.189    33
              21             0.9992 0.8665   1.34   397
              63              1.043  0.863  1.119   238
              126              1.11  1.037  1.132    64
              127             1.072  1.019  1.109    36
              128            0.9502 0.9434 0.9562    28
fx            21             0.9786 0.9348  1.053  1212
              63             0.9923 0.9058  1.279  1191
              64              1.023 0.7086  1.387   140
              65              1.011 0.9887  1.045    57
              126             1.007 0.9079  1.175   198
              129             1.009 0.9978  1.025    28
              189            0.9862  0.912   1.04   132
payrolls      21 (2 months)   1.274  1.274  1.274     1
rates         21             0.9777 0.9511  1.021   582
              63              1.002 0.9352  1.152  1167
              64              1.002 0.9513  1.078    78
              126             1.023 0.9204  1.222   809
              127            0.9527 0.9427 0.9604    25
              129             1.109  1.105  1.112    22
              189             1.048 0.9163  1.247   240
unemployment  145 (8 months)  1.156 0.9174  1.395     2
              165 (9 months)  1.167 0.9147   1.42     2

## 3 — Every model on the same backtest

In [4]:
fam = pd.Series({u: UNITS_DATA[u]["family"] for u in UNIT_IDS})
models = {"M0": M0_SCORES, "M1.1 (per asset type)": M11_SCORES,
          "production walk (text off)": pd.read_parquet(DATA / "production_backtest_scores.parquet"),
          "M1.5 (EWMA sd 63)": pd.read_parquet(DATA / "m1_5_backtest_scores.parquet"),
          "M1.7 (recent trend)": pd.read_parquet(DATA / "m1_7_backtest_scores.parquet"),
          "M1.8 (+ floor, variance ratio)": M18}
SC = {name: vs_m0(sc) for name, sc in models.items()}
LADDER = pd.DataFrame({name: {**{f: s[fam[fam == f].index].mean() for f in sorted(fam.unique())}, "all": s.mean(),
                              "better than M0": f"{(s < 1).sum()}/{len(s)}"} for name, s in SC.items()}).T
LADDER

,T2-F1,T2-F2,T2-F3,T2-F4,all,better than M0
M0,1,1,1,1,1,0/103
M1.1 (per asset type),0.9955,0.999,0.9993,0.9968,0.9976,25/103
production walk (text off),0.9288,1.011,0.9274,1.003,0.9726,74/103
M1.5 (EWMA sd 63),0.8932,0.99,0.9141,0.9642,0.9444,85/103
M1.7 (recent trend),0.8828,0.9656,0.9124,0.9505,0.9312,86/103
"M1.8 (+ floor, variance ratio)",0.8714,0.961,0.9097,0.9498,0.9266,84/103


## 4 — M1.7 vs M1.8: per family, per asset type, and the units that moved most

In [5]:
PER_UNIT = pd.DataFrame({"family": fam, "asset type": pd.Series(TYPE), "M1.5 vs M0": SC["M1.5 (EWMA sd 63)"],
                         "M1.7 vs M0": SC["M1.7 (recent trend)"], "M1.8 vs M0": SC["M1.8 (+ floor, variance ratio)"]}).loc[:, lambda d: ~d.columns.duplicated()]
def summary(g):
    return {"units": len(g), "M1.5": g["M1.5 vs M0"].mean(), "M1.7": g["M1.7 vs M0"].mean(), "M1.8": g["M1.8 vs M0"].mean(),
            "M1.8 better than M1.7": f'{(g["M1.8 vs M0"] < g["M1.7 vs M0"]).sum()}/{len(g)}',
            "M1.8 better than M0": f'{(g["M1.8 vs M0"] < 1).sum()}/{len(g)}'}
FAMILY = pd.DataFrame({f: summary(g) for f, g in [*PER_UNIT.groupby("family"), ("all", PER_UNIT)]}).T.rename_axis("family")
BY_TYPE = pd.DataFrame({t: summary(g) for t, g in PER_UNIT.groupby("asset type")}).T.rename_axis("asset type")
print("per family:"); display(FAMILY)
print("per asset type:"); display(BY_TYPE)
moved = (PER_UNIT["M1.8 vs M0"] - PER_UNIT["M1.7 vs M0"]).sort_values()
print("the units that moved most (M1.8 - M1.7; below 0 = better):")
display(PER_UNIT.loc[list(moved.index[:8]) + list(moved.index[-5:])].assign(change=moved))

per family:


,units,M1.5,M1.7,M1.8,M1.8 better than M1.7,M1.8 better than M0
family,,,,,,
T2-F1,23,0.8932,0.8828,0.8714,15/23,19/23
T2-F2,27,0.99,0.9656,0.961,11/27,18/27
T2-F3,22,0.9141,0.9124,0.9097,12/22,21/22
T2-F4,31,0.9642,0.9505,0.9498,15/31,26/31
all,103,0.9444,0.9312,0.9266,53/103,84/103


per asset type:


,units,M1.5,M1.7,M1.8,M1.8 better than M1.7,M1.8 better than M0
asset type,,,,,,
cpi,2,0.899,0.7598,0.735,2/2,2/2
equity_factor,16,0.9665,0.9708,0.9673,10/16,12/16
fx,35,0.9614,0.949,0.9447,9/35,29/35
fx+rates,5,0.933,0.9298,0.9297,3/5,5/5
payrolls,1,0.9682,0.6781,0.8225,0/1,1/1
rates,43,0.9234,0.9135,0.9059,28/43,35/43
unemployment,1,1.024,1.036,1.009,1/1,0/1


the units that moved most (M1.8 - M1.7; below 0 = better):

,family,asset type,M1.5 vs M0,M1.7 vs M0,M1.8 vs M0,change
t2-F4-hike-cycle-2021Q4b,T2-F4,rates,0.8661,0.8665,0.7769,-0.08961
t2-F2-fragile-five-brl-2013,T2-F2,fx,0.9705,0.9792,0.9053,-0.07388
t2-F2-cnh-tradewar-2018,T2-F2,fx,1.979,1.538,1.468,-0.0699
t2-F1-greater-confidence-2024,T2-F1,rates,0.9019,0.8699,0.8133,-0.05667
t2-F1-hawkish-cut-2024,T2-F1,rates,0.8734,0.8442,0.7958,-0.04839
t2-F1-third-cut-2019,T2-F1,rates,1.016,0.9922,0.9478,-0.04443
t2-F1-cpi-glidepath-2023,T2-F1,cpi,0.8749,0.7708,0.7328,-0.03806
t2-F1-liftoff-telegraph-2015,T2-F1,rates,0.8888,0.8888,0.8515,-0.03734
t2-F4-quant-crowding-2007,T2-F4,equity_factor,0.9823,0.9811,0.9938,0.01275
t2-F1-ai-mom-2024,T2-F1,equity_factor,0.7814,0.7818,0.7978,0.01591


## 5 — Save

In [6]:
M18.to_parquet(DATA / "m1_8_backtest_scores.parquet", index=False)
PER_UNIT.to_csv(DATA / "m1_8_vs_m0_per_unit.csv"); FAMILY.to_csv(DATA / "m1_8_vs_m0_per_family.csv")
BY_TYPE.to_csv(DATA / "m1_8_vs_m0_by_asset_type.csv"); LADDER.to_csv(DATA / "all_models_vs_m0.csv")
for name in ("m1_8_backtest_scores.parquet", "m1_8_vs_m0_per_unit.csv", "m1_8_vs_m0_per_family.csv", "m1_8_vs_m0_by_asset_type.csv"):
    print(f"data/{name:<30} {(DATA / name).stat().st_size / 1e3:8.1f} kB")

data/m1_8_backtest_scores.parquet      867.1 kB
data/m1_8_vs_m0_per_unit.csv             9.7 kB
data/m1_8_vs_m0_per_family.csv           0.5 kB
data/m1_8_vs_m0_by_asset_type.csv        0.6 kB


## 6 — In short

In [7]:
for name in LADDER.index:
    print(f"  {name:<36} {LADDER.loc[name, 'all']:.4f}   (better than M0 on {LADDER.loc[name, 'better than M0']})")
a = FAMILY.loc["all"]
print(f"M1.8 vs M1.7: better on {a['M1.8 better than M1.7']} units; per family M1.8 (M1.7): "
      + "; ".join(f"{f} {FAMILY.loc[f, 'M1.8']:.4f} ({FAMILY.loc[f, 'M1.7']:.4f})" for f in FAMILY.index if f != "all"))

  M0                                   1.0000   (better than M0 on 0/103)
  M1.1 (per asset type)                0.9976   (better than M0 on 25/103)
  production walk (text off)           0.9726   (better than M0 on 74/103)
  M1.5 (EWMA sd 63)                    0.9444   (better than M0 on 85/103)
  M1.7 (recent trend)                  0.9312   (better than M0 on 86/103)
  M1.8 (+ floor, variance ratio)       0.9266   (better than M0 on 84/103)
M1.8 vs M1.7: better on 53/103 units; per family M1.8 (M1.7): T2-F1 0.8714 (0.8828); T2-F2 0.9610 (0.9656); T2-F3 0.9097 (0.9124); T2-F4 0.9498 (0.9505)
